About this notebook.

This notebook goes through all the texts and aplies on them a NLP pipeline consisting of (1) cleaning of the raw text, (2) sentence tokenization, (3) part-of-speech annotation, (4) lemmatization, and (5) named entity recognition.

The processed textual data are saved for future reuse.

In [1]:
import spacy
import os
import glob
from spacy.tokens import Doc
from spacy.language import Language
import pickle
from unidecode import unidecode
import sddk
import pandas as pd
import re
import sys
import importlib
import json
from spacy.tokens import Token
from spacy.language import Language
import google_conf
import pandas as pd
import json

In [2]:
import stanza
#stanza.download("grc")

/home/jupyter-vojta/notebooks/labyrinth/venv_torch_nlp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
emlap_metadata = pd.read_csv("../data/emlap_metadata.csv", sep=";", index_col=0)
emlap_metadata.head(5)

,working_title,filenames,no.,is_done,is_noscemus,if_noscemus_id,AUTHORSHIP,is_one_author,#if more than 1 author skip section and choose compendium below,is_author_known,...,CONTENTS,genre,subject,SOURCE OF FILE,link,source_of_file,origin_of_copy,other_notes,tokens_N,aurhor_wd
0,"Augurello, Chrysopoeia",100001_Augurello1515_Chrysopoeia_GB_Noscemus,100001,True,True,713324.0,NaN,True,NaN,True,...,NaN,didactic poem,alchemy,NaN,https://wiki.uibk.ac.at/noscemus/Chrysopoeia,GB,Noscemus,NaN,23718,NaN
1,"Pseudo-Lull, Secretis",100002_Pseudo-Lull1518_De secretis_naturae_MDZ...,100002,True,False,NaN,NaN,True,NaN,True,...,NaN,treatise,"alchemy, medicine",NaN,https://www.digitale-sammlungen.de/en/view/bsb...,MDZ,MBS,NaN,24673,NaN
2,"Pantheus, Ars Transmutatione",100003_Pantheus1518_Ars_Transmutationis_Metall...,100003,True,False,NaN,NaN,True,NaN,True,...,NaN,treatise,alchemy,NaN,https://www.google.co.uk/books/edition/Ars_Tra...,GB,BL,NaN,8646,NaN
3,"Anon, Vera alchemiae",100004_Anon1561_Verae_Alchemiae_MDZ_MBS,100004,True,False,NaN,NaN,True,NaN,True,...,NaN,"compendium, florilegium",alchemy,NaN,https://mdz-nbn-resolving.de/details:bsb10141168,MDZ,MBS,NaN,3521,NaN
4,"Pantheus, Voarchadumia",100005_Pantheus1530_Voarchadumia_ONB,100005,True,False,NaN,NaN,True,NaN,True,...,NaN,treatise,alchemy,NaN,https://data.onb.ac.at/rep/10588E49,ONB,ONB,NaN,20386,NaN


In [4]:
row = emlap_metadata.loc[emlap_metadata["no."]==100001].iloc[0]

In [5]:
row["working_title"]

'Augurello, Chrysopoeia'

In [6]:
import json
import ast
import re

def parse_messy_json(text):
    if not isinstance(text, str):
        return None

    # 1. Try strict JSON first
    try:
        return json.loads(text)
    except Exception:
        pass

    # 2. Fix common issues
    fixed = text.strip()

    # Replace single quotes with double quotes when appropriate
    fixed = fixed.replace("'", '"')

    # Remove trailing commas before ] or }
    fixed = re.sub(r",\s*([}\]])", r"\1", fixed)

    # Ensure keys are quoted (naively, but works for your case)
    fixed = re.sub(r"(?<=\{|\s)([A-Za-z_][A-Za-z0-9_]*)(?=\s*:)", r'"\1"', fixed)

    # 3. Try JSON again
    try:
        return json.loads(fixed)
    except Exception:
        pass

    # 4. Try Python literal (safer than eval)
    try:
        return ast.literal_eval(text)
    except Exception:
        pass

    # 5. Give up
    return None

In [7]:
emlap_metadata["if_compendium_parsed"] = emlap_metadata["if_compendium"].apply(parse_messy_json).tolist()

In [8]:
len(emlap_metadata)

100

In [9]:
#filename_id_dict = dict(zip(emlap_metadata["filename"], emlap_metadata["No."]))

For preprocessing the latin texts, we will use a module located outside of the current repository, specifically at the same level one level up.

The module can be clonned from here: https://github.com/CCS-ZCU/latin-preprocessing and imported to python following the steps below:

In [10]:
import spacy_stanza

In [11]:
greek_nlp = spacy_stanza.load_pipeline("grc")

2025-11-26 14:10:21 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2025-11-26 14:10:22 INFO: Loading these models for language: grc (Ancient_Greek):
| Processor | Package         |
-------------------------------
| tokenize  | proiel          |
| pos       | proiel_nocharlm |
| lemma     | proiel_nocharlm |
| depparse  | proiel_nocharlm |

2025-11-26 14:10:22 INFO: Using device: cuda
2025-11-26 14:10:22 INFO: Loading: tokenize
2025-11-26 14:10:23 INFO: Loading: pos
2025-11-26 14:10:23 INFO: Loading: lemma
2025-11-26 14:10:23 INFO: Loading: depparse
2025-11-26 14:10:23 INFO: Done loading processors!


In [12]:
doc = greek_nlp('δοκῶ μοι περὶ ὧν πυνθάνεσθε οὐκ ἀμελέτητος εἶναι')

for token in doc:
   print(f'{token.text}, lemma: {token.lemma_}, pos: {token.pos_}, dep: {token.dep_}')

δοκῶ, lemma: δοκέω, pos: VERB, dep: root
μοι, lemma: ἐγώ, pos: PRON, dep: obl:arg
περὶ, lemma: περί, pos: ADP, dep: case
ὧν, lemma: ὅς, pos: PRON, dep: obj
πυνθάνεσθε, lemma: πυνθάνομαι, pos: VERB, dep: acl
οὐκ, lemma: οὐ, pos: ADV, dep: advmod
ἀμελέτητος, lemma: ἀμελέτης, pos: NOUN, dep: ccomp
εἶναι, lemma: εἰμί, pos: AUX, dep: cop


In [13]:
# for preprocessing the latin texts, we will use a module located outside of the current repository, specifically at the same level as the current project.
current_working_directory = os.getcwd()
relative_path = '../../latin-preprocessing/' # change according to your location...
module_path = os.path.abspath(os.path.join(current_working_directory, relative_path))
if module_path not in sys.path:
    sys.path.insert(0, module_path)
# Now import the module
import tomela

Using CPU for spaCy.


In [14]:
importlib.reload(tomela)

Using CPU for spaCy.


<module 'tomela' from '/home/jupyter-vojta/notebooks/latin-preprocessing/tomela/__init__.py'>

In [15]:
tomela.nlp.pipeline

[('senter', <spacy.pipeline.senter.SentenceRecognizer at 0x78e5b0887bf0>),
 ('normer', <function la_core_web_lg.functions.normer(doc)>),
 ('tok2vec', <spacy.pipeline.tok2vec.Tok2Vec at 0x78e5b0887cb0>),
 ('tagger', <spacy.pipeline.tagger.Tagger at 0x78e5b0887fb0>),
 ('morphologizer',
  <spacy.pipeline.morphologizer.Morphologizer at 0x78e5b0887dd0>),
 ('trainable_lemmatizer',
  <spacy.pipeline.edit_tree_lemmatizer.EditTreeLemmatizer at 0x78e5b0887f50>),
 ('parser', <spacy.pipeline.dep_parser.DependencyParser at 0x78e591b8c580>),
 ('lookup_lemmatizer',
  <function la_core_web_lg.functions.make_lookup_lemmatizer_function(doc)>),
 ('ner', <spacy.pipeline.ner.EntityRecognizer at 0x78e591b8fdf0>),
 ('remorpher', <function la_core_web_lg.functions.remorpher(doc)>)]

In [16]:
tomela.nlp.max_length = 4000000

In [17]:
doc = tomela.nlp("Veritas, vt vlla dicit, semper est universalis et a principiis fundamentalis oritur (lib. 3, cap. VI)")
for token in doc:
    print((token.text, token.lemma_, token.pos_))

('Veritas', 'ueritas', 'NOUN')
(',', ',', 'PUNCT')
('vt', 'vt', 'ADV')
('vlla', 'vllus', 'NOUN')
('dicit', 'dico', 'VERB')
(',', ',', 'PUNCT')
('semper', 'semper', 'ADV')
('est', 'sum', 'AUX')
('universalis', 'uniuersalis', 'ADJ')
('et', 'et', 'CCONJ')
('a', 'ab', 'ADP')
('principiis', 'principium', 'NOUN')
('fundamentalis', 'fundamentalis', 'ADJ')
('oritur', 'orior', 'VERB')
('(lib', '(lib', 'NOUN')
('.', '.', 'PUNCT')
('3', '3', 'NUM')
(',', ',', 'PUNCT')
('cap', 'capitulum', 'NOUN')
('.', '.', 'PUNCT')
('VI', 'uis', 'NUM')
(')', ')', 'PUNCT')


In [18]:
source_path = "/srv/data/tome/tome-corpus/EMLAP_2025-10-31/annotated_textblocks/"
len(os.listdir(source_path))

200

In [19]:
import shutil
shutil.copytree("/srv/data/tome/tome-corpus/EMLAP_2025-10-31/annotated_textblocks/", "../data/emlap_annotated_textblocks/", dirs_exist_ok=True)

'../data/emlap_annotated_textblocks/'

In [20]:
[f for f in sorted(os.listdir(source_path)) if "_params" not in f]

['100001_Augurello1515_Chrysopoeia_GB_Noscemus.json',
 '100002_Pseudo-Lull1518_De_secretis_naturae_MDZ_MBS.json',
 '100003_Pantheus1518_Ars_Transmutationis_Metallicae_BL_GB.json',
 '100004_Anon1561_Verae_Alchemiae_MDZ_MBS.json',
 '100005_Pantheus1530_Voarchadumia_ONB.json',
 '100006_Savonarola1532_De_arte_conficiendi_aquam_vitae_ONB.json',
 '100007_Anon1550_Rosarium_philosophorum_ER_ZZ.json',
 '100008_Severinus1572_Epistola_MBZ_MBS.json',
 '100009_Vegius1518_Inter_inferiora_corpora_disputatio_ONB.json',
 '100010_Bracesco1548_De_alchemia_dialogi_duo_IA_Madrid.json',
 '100011_Anon1541_De_alchemia_MDZ_MBS.json',
 '100012_Gessner1552_Thesaurus_Euonymi_Philiatri_ER_ZZ.json',
 '100013_Ulstad1525_Coelum_philosophorum_Medica_BIUSP.json',
 '100014_Toxites1567_Spongia_stibii_MDZ_MBS.json',
 '100015_Gessner1569_Thesaurus_Euonymi_Philiatri_liber_secundus_MDZ_MBS.json',
 '100016_Bonus1546_Pretiosa_Margarita_Novella_ONB.json',
 '100017_Bodenstein1559_Isagoge_MDZ_MBS.json',
 '100018_Trevisanus1567_Pe

In [21]:
row = emlap_metadata.loc[emlap_metadata["no."]==int("100001")].iloc[0]
row

working_title                                                                                 Augurello, Chrysopoeia
filenames                                                               100001_Augurello1515_Chrysopoeia_GB_Noscemus
no.                                                                                                           100001
is_done                                                                                                         True
is_noscemus                                                                                                     True
if_noscemus_id                                                                                              713324.0
AUTHORSHIP                                                                                                       NaN
is_one_author                                                                                                   True
#if more than 1 author skip section and choose compendium below 

In [22]:
files_overview = []
for filename in os.listdir(source_path):
    #filename = 'DuChesne1575_Ad_Iacobi_Auberti_MDZ_Augsburg.json'
    id = int(filename[:6])
    row = emlap_metadata.loc[emlap_metadata["no."]==id].iloc[0]
    if "_params" not in filename:
        filepath = os.path.join(source_path, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            textblocks = json.load(f)
        pages_n = len(textblocks)
        chars_n = sum([sum([len(tb["text"]) for tb in p]) for p in textblocks])
        files_overview.append({"filename" : filename, "pages_n" : pages_n, "chars_n" : chars_n})
files_processed = pd.DataFrame(files_overview)
files_processed

,filename,pages_n,chars_n
0,100084_Croll1609_Basilica_chymica_MDZ_MBS.json,477,694793
1,100094_Anon1625_Musaeum_hermeticum_VD17_SLUB.json,508,694493
2,100058_Hagecius1596_Actio_medica_ER_UBB.json,89,104677
3,100013_Ulstad1525_Coelum_philosophorum_Medica_...,113,237793
4,100069_Severinus1571_Idea_medicinae_philosophi...,463,600046
...,...,...,...
95,100056_Claveus1598_Apologia_crysopoeiae_MDZ_MB...,233,233328
96,100065_Libavius1594_Neoparacelsica_MDZ_MBS.json,821,1079977
97,100031_Phaedro1562_Aquila_coelestis_MBZ_MBS.json,55,15562
98,100098_Burggravius1630_Biolychnium_VD17_SLUB.json,167,186267


In [23]:
#emlap_catalogue = google_conf.setup(sheet_url="https://docs.google.com/spreadsheets/d/1bkHHTYc86K2IuEXqfYfkDNt5LovtvCU3gvqHIbVio88/edit?usp=sharing", service_account_path="../../../ServiceAccountsKey.json")

# google_conf.set_with_dataframe(emlap_catalogue.add_worksheet("files_processed", 1,1), files_processed, include_index=False)


Develop and test with one example test

In [64]:
filename = '100085_Libavius1606_Commentariorum_alchemiae_pars_1_MDZ_MBS.json'
filepath = os.path.join(source_path, filename)
with open(filepath, 'r', encoding='utf-8') as f:
    textblocks = json.load(f)

In [65]:
textblocks_unheadered = []
for p in textblocks:
    p_unheadered = []
    header_met = False
    for textblock in p:
        if textblock["tag"] == "header":
            if header_met:
                textblock["tag"] = "text"
            else:
                header_met = True
        p_unheadered.append(textblock)
    textblocks_unheadered.append(p_unheadered)

textblocks = textblocks_unheadered#%%
len(textblocks)

413

In [66]:
textblocks[30][:10]

[{'coordinates': [211.67999267578125,
   1086.2401123046875,
   361.0691833496094,
   1091.0400390625],
  'text': '6. Aeneid.\n',
  'tag': 'margin'},
 {'coordinates': [405.6000061035156,
   200.16000366210938,
   1988.2835693359375,
   217.34410095214844],
  'text': '20\nExamen sententiae Parisiensis scholae\n',
  'tag': 'header'},
 {'coordinates': [573.3599853515625,
   284.6640930175781,
   2086.989501953125,
   289.5841064453125],
  'text': '"Si in mundo sublunari nulla est substantia ab elementaribus quatuor distincta, nulla est quinta essen¬\n',
  'tag': 'text'},
 {'coordinates': [672.47998046875,
   334.7998962402344,
   1029.914794921875,
   339.5998840332031],
  'text': 'tia, quae extrahi possit.\n',
  'tag': 'text'},
 {'coordinates': [573.3599853515625,
   384.9600524902344,
   1760.6229248046875,
   389.7600402832031],
  'text': 'Prius est. Posterius ergo: & per consequens, ignis & opera perditur extrahendo,"\n',
  'tag': 'text'},
 {'coordinates': [504.9599914550781,
   433.6

In [67]:
margin_pages = []
for p in textblocks[:50]:
    for textblock in p:
        if textblock["tag"] == "margin":
            margin_pages.append(p)
            break

In [68]:
len(margin_pages)

35

In [69]:
margin_pages[0]

[{'coordinates': [235.44000244140625,
   684.7200317382812,
   409.4384460449219,
   689.5200805664062],
  'text': 'In specie tot\n',
  'tag': 'margin'},
 {'coordinates': [235.44000244140625,
   725.0398559570312,
   410.12158203125,
   729.8399047851562],
  'text': 'medicorum\n',
  'tag': 'margin'},
 {'coordinates': [235.44000244140625,
   765.8640747070312,
   409.9117736816406,
   770.7840576171875],
  'text': 'hodie sectae,\n',
  'tag': 'margin'},
 {'coordinates': [240.24000549316406,
   806.3999633789062,
   399.8396301269531,
   811.2000122070312],
  'text': 'vt videan¬\n',
  'tag': 'margin'},
 {'coordinates': [237.83999633789062,
   846.2639770507812,
   409.64459228515625,
   851.1839599609375],
  'text': 'tur innume¬\n',
  'tag': 'margin'},
 {'coordinates': [237.83999633789062,
   885.8640747070312,
   410.22802734375,
   890.7840576171875],
  'text': 'rabila, gene¬\n',
  'tag': 'margin'},
 {'coordinates': [237.83999633789062,
   929.0398559570312,
   394.5736389160156,
   933

In [70]:
margin_pages[1]

[{'coordinates': [526.0800170898438,
   172.10403442382812,
   1639.3897705078125,
   177.02403259277344],
  'text': 'EPISTOLA DEDICATORIA.\n',
  'tag': 'header'},
 {'coordinates': [337.9200134277344,
   262.1040344238281,
   1993.8466796875,
   267.0240478515625],
  'text': 'Nullus quidem Paracelsita quicquam sibi commune cum schola\n',
  'tag': 'text'},
 {'coordinates': [194.63999938964844,
   338.9038391113281,
   1991.999755859375,
   343.8238525390625],
  'text': 'Galenica esse prae se videtur ferre: aliqui tamen Hippocrati sunt aequio¬\n',
  'tag': 'text'},
 {'coordinates': [181.44000244140625,
   417.8399353027344,
   1995.948486328125,
   422.6399230957031],
  'text': 'res: Hic reijcit Chrysopoeian, ille magian, adiurationes, execrationes ec.\n',
  'tag': 'text'},
 {'coordinates': [185.75999450683594,
   495.3838195800781,
   1972.9329833984375,
   500.3038330078125],
  'text': 'Puri puti Paracelsici omnia atque etiam sputa sui Monarchae exosculan¬\n',
  'tag': 'text'},
 {'coor

In [71]:
for p in textblocks[:50]:
    for t in p:
        if "[" in t["text"]:
            print(t)

{'coordinates': [384.0, 2262.9599609375, 2165.489013671875, 2267.760009765625], 'text': '[GR]χιμικώτερα[/GR] essent seligere, dissentientiaquè componere. Itaque & nouam\n', 'tag': 'text'}
{'coordinates': [139.1999969482422, 3188.159912109375, 1914.183349609375, 3192.9599609375], 'text': 'quoddam [GR]μιγμα[/GR] & commentum ab omni veritate alienum. Sed ostendendum est veram artem nec\n', 'tag': 'text'}
{'coordinates': [398.1600036621094, 511.6800231933594, 2167.622314453125, 516.4800415039062], 'text': 'longe abest à [GR]πανσπερμία[/GR] illa vetere, & atomis Democriti, & multiplici vicissim oriente & pereunte? Sed\n', 'tag': 'text'}
{'coordinates': [390.7200012207031, 2782.079833984375, 2149.680419921875, 2786.8798828125], 'text': 'quętis [GR]σκευασταί καὶ πυροτεηνίαν[/GR], ecce tibi eundem Bulcasim passim inseruientem. Fornaces enim seu Atha¬\n', 'tag': 'text'}
{'coordinates': [198.24000549316406, 1149.1199951171875, 1980.812744140625, 1153.919921875], 'text': 'potest. "Nec ego (inquit

In [72]:
for p in textblocks:
    for t in p:
        if "[/S]" in t["text"]:
            print(t)

{'coordinates': [225.83999633789062, 905.7598266601562, 387.3038635253906, 910.5598754882812], 'text': 'Olei [S]y[/S]. j. ex\n', 'tag': 'margin'}
{'coordinates': [429.1199951171875, 1108.5599365234375, 2191.517578125, 1113.35986328125], 'text': 'crassos. Dosis eius solius [S]z[/S] ij. vsque ad [S]z[/S] iiij. cum aqua frigida. Similia habet Mesues lib simplicium, cap. 16.\n', 'tag': 'text'}
{'coordinates': [241.9199981689453, 2369.999755859375, 407.46197509765625, 2374.7998046875], 'text': 'trahe [S][/S], ex\n', 'tag': 'margin'}
{'coordinates': [234.72000122070312, 2417.999755859375, 407.2383728027344, 2422.7998046875], 'text': 'hoc\xa0[S][/S], & i¬\n', 'tag': 'margin'}
{'coordinates': [424.0799865722656, 312.7200622558594, 2143.775634765625, 317.5200500488281], 'text': 'Lunam, Venerem, Martem, Saturnum, louem; signare vero more astrologorum,\xa0[S]Mercurius[/S],\xa0[S]Sol[/S],\xa0[S]Luna[/S],\xa0[S]Venus[/S],\xa0[S]Mars[/S],\xa0[S]Saturnus[/S],\xa0[S]Jupiter[/S].\n', 'tag': 'text'}
{'c

In [73]:
def add_tag_info_to_char_mapping(full_text, char_to_source):
    """
    Parse inline tags like [GR]...[/GR], [M]...[/M], [S]...[/S] in full_text
    and attach a "tags" field (list of active tags) to char_to_source[char_idx].
    """
    active_tags = []  # stack / list of currently open tags
    i = 0
    n = len(full_text)

    while i < n:
        ch = full_text[i]

        if ch == "[":
            # Try to find the closing bracket of this [TAG] or [/TAG]
            close = full_text.find("]", i + 1)
            if close == -1:
                # malformed tag, treat as normal char
                if i in char_to_source and active_tags:
                    tags_set = char_to_source[i].setdefault("tags", set())
                    tags_set.update(active_tags)
                i += 1
                continue

            content = full_text[i + 1:close]

            if content.startswith("/"):
                # closixng tag, e.g. [/GR]
                label = content[1:]
                # remove from active_tags (LIFO or first match)
                if active_tags and active_tags[-1] == label:
                    active_tags.pop()
                elif label in active_tags:
                    active_tags.remove(label)
                # We don't assign tags to the '['...']' characters themselves here
            else:
                # opening tag, e.g. [GR]
                label = content
                active_tags.append(label)

            # skip past the closing ']'
            i = close + 1
            continue

        else:
            # normal character: assign currently active tags (if any)
            if i in char_to_source and active_tags:
                tags_set = char_to_source[i].setdefault("tags", set())
                tags_set.update(active_tags)
            i += 1

    # normalize sets → lists (so they are JSON-serializable, nicer to inspect)
    for info in char_to_source.values():
        if "tags" in info and isinstance(info["tags"], set):
            info["tags"] = sorted(info["tags"])

    return char_to_source

In [74]:
    for page_idx, page in enumerate(textblocks):
        for tb_idx, tb in enumerate(page):
            if tb["tag"] not in {"text", "title", "margin"}:
                continue



    for page_idx, page in enumerate(textblocks):
        for tb_idx, tb in enumerate(page):
            if tb["tag"] == "margin":
                continue

In [75]:
import re
import unicodedata
from spacy.tokens import Token, Doc
from spacy.language import Language

# -------------------------------------------------------------------
# 0. Token / Doc extensions
# -------------------------------------------------------------------
for ext, default in [
    ("pages", None),
    ("textblocks", None),
    ("tags", None),
    ("block_type", None),
]:
    if not Token.has_extension(ext):
        Token.set_extension(ext, default=default)

if not Doc.has_extension("char_to_source"):
    Doc.set_extension("char_to_source", default=None)


# -------------------------------------------------------------------
# CLEANER: keep tag *content*, strip tag markers, clean only outside spans
# -------------------------------------------------------------------
TAG_PATTERN = re.compile(r"(\[[A-Za-z]+]|\[/[A-Za-z]+])")


def text_cleaner_keep_tags(raw_text: str) -> str:
    """
    Clean Latin text but NEVER modify text inside markup spans like:
        [GR]...[/GR], [S]...[/S], [M]...[/M]

    Behaviour:
      - Split raw text into TAG / NON-TAG parts.
      - Maintain a stack of open tags (supports nested, though unlikely).
      - REMOVE tag markers [TAG] and [/TAG] from the output.
      - Inside-tag content is preserved exactly (NFC-normalized).
      - Outside-tag content is cleaned with the Latin OCR pipeline.
    """
    # normalize non-breaking spaces to normal spaces
    raw_text = raw_text.replace("\xa0", " ")

    parts = re.split(TAG_PATTERN, raw_text)
    cleaned_parts = []
    stack = []  # open tag names

    for part in parts:
        # Opening tag: [GR], [S], [M], ...
        if re.fullmatch(r"\[[A-Za-z]+]", part):
            stack.append(part[1:-1])
            # DO NOT append the tag itself to output
            continue

        # Closing tag: [/GR], [/S], ...
        if re.fullmatch(r"\[/[A-Za-z]+]", part):
            name = part[2:-1]
            if stack and stack[-1] == name:
                stack.pop()
            # DO NOT append the tag marker either
            continue

        # Inside any tag span → preserve content, just normalize
        if stack:
            cleaned_parts.append(unicodedata.normalize("NFC", part))
            continue

        # Outside tags → Latin cleaning
        x = part
        x = x.replace("¬\n", "").replace("\n", " ")
        x = x.replace("ß", "ss").replace("ij", "ii")
        x = re.sub(r"\s\s+", " ", x)

        # Fix OCR margin stars: "*Nota" → "Nota"
        x = re.sub(r"^\*+(\w)", r"\1", x)

        # First letter as-is, rest lowercase
        x = re.sub(r'\b(\w)(\w*)\b', lambda m: m.group(1) + m.group(2).lower(), x)

        x = x.replace(". &", ", &")
        x = x.replace("v", "u").replace("V", "U")

        cleaned_parts.append(x)

    return "".join(cleaned_parts)


# -------------------------------------------------------------------
# 1. Build raw + clean text and mapping  (generic for any tag set)
# -------------------------------------------------------------------
def _process_textblocks_for_tags(textblocks, allowed_tags):
    """
    Same logic as your original process_textblocks, but restricted to
    a set of block tags (e.g. {"text", "title"} or {"margin"}).

    Returns:
      raw_full, clean_full, raw_to_clean, char_src
    """

    raw_full = ""
    clean_full = ""
    raw_to_clean = {}
    char_src = {}

    for page_idx, page in enumerate(textblocks):
        for tb_idx, tb in enumerate(page):
            if tb["tag"] not in allowed_tags:
                continue

            raw_text = tb["text"]
            clean_text = text_cleaner_keep_tags(raw_text)

            r0 = len(raw_full)
            c0 = len(clean_full)

            r_i = 0
            c_i = 0
            len_raw = len(raw_text)
            len_clean = len(clean_text)

            while r_i < len_raw and c_i < len_clean:
                rc = raw_text[r_i]
                cc = clean_text[c_i]

                raw_idx = r0 + r_i
                clean_idx = c0 + c_i

                if rc == cc:
                    # Normal 1:1 character
                    raw_to_clean[raw_idx] = clean_idx
                    char_src[clean_idx] = {
                        "page_idx": page_idx,
                        "textblock_idx": tb_idx,
                        "textblock_type": tb["tag"],
                    }
                    r_i += 1
                    c_i += 1
                    continue

                # Raw is at a tag marker? Skip it entirely in raw.
                if raw_text.startswith("[", r_i):
                    end = raw_text.find("]", r_i)
                    if end == -1:
                        # Malformed tag: treat '[' as a normal char
                        raw_to_clean[raw_idx] = clean_idx
                        char_src[clean_idx] = {
                            "page_idx": page_idx,
                            "textblock_idx": tb_idx,
                            "textblock_type": tb["tag"],
                        }
                        r_i += 1
                        c_i += 1
                    else:
                        for skip in range(r_i, end + 1):
                            raw_to_clean[r0 + skip] = None
                        r_i = end + 1
                    continue

                # Otherwise: cleaner changed the char (case, diacritics, space, …)
                raw_to_clean[raw_idx] = clean_idx
                char_src[clean_idx] = {
                    "page_idx": page_idx,
                    "textblock_idx": tb_idx,
                    "textblock_type": tb["tag"],
                }
                r_i += 1
                c_i += 1

            # Any trailing raw chars (should be rare) → no clean mapping
            while r_i < len_raw:
                raw_to_clean[r0 + r_i] = None
                r_i += 1

            raw_full += raw_text
            clean_full += clean_text

    return raw_full, clean_full, raw_to_clean, char_src


# -------------------------------------------------------------------
# 2. Greek-aware + S-aware tag extractor
# -------------------------------------------------------------------
GREEK_RE = re.compile(r"[\u0370-\u03FF\u1F00-\u1FFF]")


def extract_tags_from_raw(raw_text: str):
    """
    Build char-level tag assignments on RAW indices.

    GR:
      - Any character in Greek Unicode ranges → tag "GR".

    S:
      - [S] ... [/S] → all characters between them get tag "S".
      - [S][/S] (empty) → ignored.
      - Lone [S] with no [/S] → tag immediately following token:
            characters from end-of-[S] up to next whitespace or '['.
    """

    raw_tags = {}  # raw_idx -> set(labels)
    markup_spans = []

    tag_pattern = re.compile(r"\[(?P<tag>[A-Za-z]+)]|\[/(?P<etag>[A-Za-z]+)]")
    matches = list(tag_pattern.finditer(raw_text))

    # record markup spans (for debugging / future use)
    for m in matches:
        markup_spans.append(m.span())

    # --- GR: any Greek character in raw_text ---
    for idx, ch in enumerate(raw_text):
        if GREEK_RE.match(ch):
            raw_tags.setdefault(idx, set()).add("GR")

    # --- S: explicit [S] ... [/S] and lone [S] ---

    # Track opening [S] tokens
    open_S = []  # list of match objects for [S]

    for m in matches:
        start, end = m.span()
        tag = m.group("tag")
        etag = m.group("etag")

        if tag == "S":
            # Opening [S]
            open_S.append(m)

        elif etag == "S":
            # Closing [/S] – match with the last unmatched [S]
            for j in range(len(open_S) - 1, -1, -1):
                open_m = open_S[j]
                if open_m is None:
                    continue
                open_end = open_m.end()
                close_start = start
                # assign S to chars between [S] and [/S]
                for raw_idx in range(open_end, close_start):
                    raw_tags.setdefault(raw_idx, set()).add("S")
                open_S[j] = None  # mark as matched
                break

    # Handle lone [S] (without a closing [/S])
    for open_m in open_S:
        if open_m is None:
            continue
        i = open_m.end()  # position just after ']'
        # Tag immediately following "token": chars until whitespace or new tag
        n = len(raw_text)
        while i < n and not raw_text[i].isspace():
            if raw_text[i] == "[":
                break  # new tag, stop
            raw_tags.setdefault(i, set()).add("S")
            i += 1

    return raw_tags, markup_spans


# -------------------------------------------------------------------
# 3. raw → clean tag mapping
# -------------------------------------------------------------------
def map_raw_tags_to_clean(raw_tags, raw_to_clean):
    clean_tags = {}  # clean_idx -> set(labels)
    for raw_idx, tagset in raw_tags.items():
        clean_idx = raw_to_clean.get(raw_idx)
        if clean_idx is not None:
            clean_tags.setdefault(clean_idx, set()).update(tagset)
    return clean_tags


# -------------------------------------------------------------------
# 4. High-level processor (MAIN + MARGINS)
# -------------------------------------------------------------------
def process_with_source_tracking(textblocks, nlp):
    """
    Returns two docs with identical structure but different content sources:

      doc_main    – based on block tags {"text", "title"}
      doc_margins – based on block tag {"margin"}

    Greek/S tags and char_to_source mappings are computed independently
    for each branch, so all indices remain consistent.
    """

    # ----- MAIN TEXT (text + title) -----
    raw_main, clean_main, raw2clean_main, src_main = _process_textblocks_for_tags(
        textblocks, allowed_tags={"text", "title"}
    )

    raw_tags_main, _ = extract_tags_from_raw(raw_main)
    clean_tags_main = map_raw_tags_to_clean(raw_tags_main, raw2clean_main)

    for clean_idx, tagset in clean_tags_main.items():
        if clean_idx in src_main:
            existing = set(src_main[clean_idx].get("tags", []))
            src_main[clean_idx]["tags"] = sorted(existing | set(tagset))

    doc_main = nlp.make_doc(clean_main)
    doc_main._.char_to_source = src_main

    for name, proc in nlp.pipeline:
        doc_main = proc(doc_main)

    # ----- MARGINS ONLY -----
    raw_marg, clean_marg, raw2clean_marg, src_marg = _process_textblocks_for_tags(
        textblocks, allowed_tags={"margin"}
    )

    raw_tags_marg, _ = extract_tags_from_raw(raw_marg)
    clean_tags_marg = map_raw_tags_to_clean(raw_tags_marg, raw2clean_marg)

    for clean_idx, tagset in clean_tags_marg.items():
        if clean_idx in src_marg:
            existing = set(src_marg[clean_idx].get("tags", []))
            src_marg[clean_idx]["tags"] = sorted(existing | set(tagset))

    doc_marg = nlp.make_doc(clean_marg)
    doc_marg._.char_to_source = src_marg

    for name, proc in nlp.pipeline:
        doc_marg = proc(doc_marg)

    return doc_main, doc_marg


# -------------------------------------------------------------------
# 5. Source tracker: assign page / textblock / tags / block_type to tokens
# -------------------------------------------------------------------
@Language.component("source_tracker")
def source_tracker(doc):
    cts = doc._.char_to_source
    if cts is None:
        return doc

    for token in doc:
        pages = set()
        textblocks = set()
        block_types = set()
        tags = set()

        for i in range(token.idx, token.idx + len(token.text)):
            info = cts.get(i)
            if not info:
                continue
            pages.add(info["page_idx"])
            textblocks.add(info["textblock_idx"])
            bt = info.get("textblock_type")
            if bt:
                block_types.add(bt)
            if "tags" in info:
                tags.update(info["tags"])

        token._.pages = sorted(pages) if pages else None
        token._.textblocks = sorted(textblocks) if textblocks else None
        token._.block_type = next(iter(block_types)) if block_types else "text"
        token._.tags = sorted(tags) if tags else None

    return doc


# -------------------------------------------------------------------
# 6. Block-type-based sentence splitter
# -------------------------------------------------------------------
@Language.component("blocktype_sentencizer")
def blocktype_sentencizer(doc):
    prev_bt = None
    for token in doc:
        bt = token._.block_type
        if prev_bt is None or bt != prev_bt:
            token.is_sent_start = True
        prev_bt = bt
    return doc


# -------------------------------------------------------------------
# 7. Greek enrichment (unchanged logic, applied to both docs)
# -------------------------------------------------------------------
for ext in ["gr_lemma", "gr_pos", "gr_morph", "gr_dep", "gr_norm"]:
    if not Token.has_extension(ext):
        Token.set_extension(ext, default=None)


@Language.component("greek_enricher")
def greek_enricher(doc):
    """
    Post-hoc Greek morphological enrichment:

      - Find contiguous GR-tagged spans in the doc.
      - For each span, run greek_nlp(span.text).
      - Align tokens by order (zip).
      - Copy Greek lemma / POS / morph / dep / norm into:
          - custom attrs: ._.gr_*
          - and override Latin .lemma_ and .pos_ for GR tokens.
    """

    greek_spans = []
    current = []

    # 1) Group contiguous GR tokens
    for tok in doc:
        if tok._.tags and "GR" in tok._.tags:
            current.append(tok)
        else:
            if current:
                greek_spans.append(current)
                current = []
    if current:
        greek_spans.append(current)

    if not greek_spans:
        return doc

    # 2) Process each GR span with greek_nlp
    for span_tokens in greek_spans:
        span_start = span_tokens[0].i
        span_end   = span_tokens[-1].i + 1
        span = doc[span_start:span_end]

        gdoc = greek_nlp(span.text)
        g_tokens = list(gdoc)

        # 3) Align by order; guard against length mismatches
        for latin_tok, g_tok in zip(span, g_tokens):
            if not (latin_tok._.tags and "GR" in latin_tok._.tags):
                continue

            latin_tok._.gr_lemma = g_tok.lemma_
            latin_tok._.gr_pos   = g_tok.pos_
            latin_tok._.gr_morph = g_tok.morph.to_dict()
            latin_tok._.gr_dep   = g_tok.dep_
            latin_tok._.gr_norm  = g_tok.norm_

            # override the Latin analysis
            latin_tok.lemma_ = g_tok.lemma_
            latin_tok.pos_   = g_tok.pos_

    return doc


# -------------------------------------------------------------------
# 8. Register components into your existing pipeline
# -------------------------------------------------------------------
if "source_tracker" in tomela.nlp.pipe_names:
    tomela.nlp.remove_pipe("source_tracker")
tomela.nlp.add_pipe("source_tracker", before="senter")

if "blocktype_sentencizer" in tomela.nlp.pipe_names:
    tomela.nlp.remove_pipe("blocktype_sentencizer")
tomela.nlp.add_pipe("blocktype_sentencizer", after="source_tracker")

if "greek_enricher" in tomela.nlp.pipe_names:
    tomela.nlp.remove_pipe("greek_enricher")
tomela.nlp.add_pipe("greek_enricher", last=True)

<function __main__.greek_enricher(doc)>

In [76]:
doc, doc_margins = process_with_source_tracking(textblocks[:20], tomela.nlp)

In [77]:
for t in doc:
    if t._.tags and "GR" in t._.tags:
        print(
            "TAG:",
            repr(t.text),
            "lemma=", t.lemma_,
            "pos=", t.pos_,
            "gr_lemma=", t._.gr_lemma,
            "gr_pos=", t._.gr_pos,
            "tags=", t._.tags,
        )

TAG: 'χιμικωτερα' lemma= χιμικός pos= ADJ gr_lemma= χιμικός gr_pos= ADJ tags= ['GR']
TAG: 'μιγμα' lemma= μιγμα pos= INTJ gr_lemma= μιγμα gr_pos= INTJ tags= ['GR']
TAG: 'πανσπερμια' lemma= πανσπερμια pos= INTJ gr_lemma= πανσπερμια gr_pos= INTJ tags= ['GR']
TAG: 'σκευασται' lemma= σκευαά pos= NOUN gr_lemma= σκευαά gr_pos= NOUN tags= ['GR']
TAG: 'και' lemma= και pos= NOUN gr_lemma= και gr_pos= NOUN tags= ['GR']
TAG: 'πυροτεηνιαν' lemma= πυροτεηνιανή pos= NOUN gr_lemma= πυροτεηνιανή gr_pos= NOUN tags= ['GR']
TAG: 'κακοηθειαν' lemma= κακοήθειος pos= ADJ gr_lemma= κακοήθειος gr_pos= ADJ tags= ['GR']


In [78]:
for sent in doc_margins.sents:
    print(sent.text)

In specie tot medicorum hodie sectae, ut uideantur innumerabila, genere ad tres classes reduci possunt.
Galenici ingenui.
Chymiatri uulgares seu simplices.
Chymiatri hermetici.
Chymiatri.
sophistici, non parum paracelsici, licet hoc nomen auersentur.
Talis fuit Assyluanus.
Paracelsici.
Hemiparacelsici.
Heloparacelsici.
Acto. 7. 22.
i.
metaph.
iux. 19.
Pag. 65.
apol.
pro Hipp.
Iulius Firmicus.
Albucasis Mesues ex regum Damascifamilia plaerumque uixisse.
narratur circa annum 155.
& 1158.
AliGodefrido Coatanem faciunt an.
1099.
Alchymia, aetate Mesue celebris cum suis artificibus.
Uidec. 8. 4.
Tamarindis.
c. 10.
de rosis c. 12 simp.
de absinth.
ubi Marini annotatio Chymicis honorifica.c. 14.
de fumaria.
c. 1 lib. 1.
de med.
laboriose soluentibus Item c. 3. 7. 9.
Ii. 13. 14. 22.
etc.
Albert.
Magnus.
Thom.
de Aquin.
Arnold.
de Uilla noua, Reimundus Lullius.
Auicennos Ita Ualescus de Taranta, qui scripsit an.
Ch. 1418.
fuisset prior Auicenna, etc Auicenna, Auerrhoes, Abenzoar, an. 1149.
temp

In [79]:
def doc_to_sent_dicts(doc, work_id):
    """
    Convert a spaCy Doc into the final v2 JSON sentence structure.
    Includes Greek overrides, lemma fallback, and full metadata.
    """

    sent_dicts = []

    for sent_id, sent in enumerate(doc.sents):

        sent_tokens = []
        sent_start = sent.start_char

        for t in sent:

            # --- Greek override ---
            if t._.tags and "GR" in t._.tags:
                lemma = t._.gr_lemma or t.lemma_
                pos   = t._.gr_pos   or t.pos_
            else:
                lemma = t.lemma_
                pos   = t.pos_

            # lemma fallback
            if not lemma:
                lemma = t.text

            # dict representation
            tok_dict = {
                "token_text": t.text,
                "lemma": lemma,
                "pos": pos,
                "ref": {
                    "page": t._.pages,
                    "textblock": t._.textblocks,
                    "tags": t._.tags,
                    "blocktype": t._.block_type,
                },
                "char_start": t.idx - sent_start,
                "char_end": (t.idx - sent_start) + len(t),
            }

            sent_tokens.append(tok_dict)

        # whole sentence
        sent_dict = {
            "work_id": work_id,
            "sent_id": sent_id,
            "sent_text": sent.text,
            "tokens_data": sent_tokens,
        }

        sent_dicts.append(sent_dict)

    return sent_dicts

In [82]:
work_id = filename[:6]
sent_dicts = doc_to_sent_dicts(doc, work_id)
sent_dicts_margins = doc_to_sent_dicts(doc_margins, work_id)

In [49]:
sent_dicts_margins[:5]

[{'work_id': '100085',
  'sent_id': 0,
  'sent_text': 'In specie tot medicorum hodie sectae, ut uideantur innumerabila, genere ad tres classes reduci possunt.',
  'tokens_data': [{'token_text': 'In',
    'lemma': 'in',
    'pos': 'ADP',
    'ref': {'page': [4],
     'textblock': [0],
     'tags': None,
     'blocktype': 'margin'},
    'char_start': 0,
    'char_end': 2},
   {'token_text': 'specie',
    'lemma': 'species',
    'pos': 'NOUN',
    'ref': {'page': [4],
     'textblock': [0],
     'tags': None,
     'blocktype': 'margin'},
    'char_start': 3,
    'char_end': 9},
   {'token_text': 'tot',
    'lemma': 'tot',
    'pos': 'DET',
    'ref': {'page': [4],
     'textblock': [0],
     'tags': None,
     'blocktype': 'margin'},
    'char_start': 10,
    'char_end': 13},
   {'token_text': 'medicorum',
    'lemma': 'medicus',
    'pos': 'NOUN',
    'ref': {'page': [4],
     'textblock': [1],
     'tags': None,
     'blocktype': 'margin'},
    'char_start': 14,
    'char_end': 23},
   

In [54]:
def get_token_coordinates(token, textblocks):
    """
    Returns the coordinates (x0, y0, x1, y1) of the ORIGINAL OCR textblock
    that produced the token.

    token.ref MUST contain:
        - "page": [int]
        - "textblock": [int]
    """

    ref = token["ref"]
    if ref is None:
        return None

    page_list = ref.get("page")
    tb_list = ref.get("textblock")

    if not page_list or not tb_list:
        return None

    page_idx = page_list[0]
    tb_idx = tb_list[0]

    try:
        tb = textblocks[page_idx][tb_idx]
        return tb["coordinates"]
    except Exception as e:
        print("Coordinate lookup failed:", e)
        return None

def attach_coordinates_to_sent_dicts(sent_dicts, textblocks):
    for sent in sent_dicts:
        for tok in sent["tokens_data"]:
            tok["coordinates"] = get_token_coordinates(tok, textblocks)
    return sent_dicts

In [55]:
sent_dicts_with_coords = attach_coordinates_to_sent_dicts(sent_dicts, textblocks)
sent_dicts_margins_with_coords = attach_coordinates_to_sent_dicts(sent_dicts_margins, textblocks)

In [ ]:
textblocks_unheadered = []
for p in textblocks:
    p_unheadered = []
    header_met = False
    for textblock in p:
        if textblock["tag"] == "header":
            if header_met:
                textblock["tag"] = "text"
            else:
                header_met = True
        p_unheadered.append(textblock)
    textblocks_unheadered.append(p_unheadered)
textblocks = textblocks_unheadered

In [63]:
[textblock for textblock in textblocks[4] if textblock["tag"] in ["header", "title", "text"]]

[{'coordinates': [752.6400146484375,
   171.59994506835938,
   1860.423095703125,
   176.3999481201172],
  'text': 'EPISTOLA DEDICATORIA.\n',
  'tag': 'header'},
 {'coordinates': [420.0,
   255.35995483398438,
   2208.9599609375,
   260.1599426269531],
  'text': 'test ex sequentibus intelligi. Triplex hodiè est medicinae docendae, facien¬\n',
  'tag': 'header'},
 {'coordinates': [424.79998779296875,
   334.7998962402344,
   2193.759765625,
   339.5998840332031],
  'text': 'daeque & ratio & familia. Primam appellant sectam dogmaticorum &\n',
  'tag': 'text'},
 {'coordinates': [420.0,
   408.9839172363281,
   2192.752685546875,
   413.9039306640625],
  'text': 'rationalium, cuius professores ab Hippocrate & Galeno Hippocratici &\n',
  'tag': 'text'},
 {'coordinates': [434.6400146484375,
   490.5599060058594,
   2202.1669921875,
   495.3598937988281],
  'text': 'Galenici nuncupantur, quia potissmum, istorum, instituta sequuntur.\n',
  'tag': 'text'},
 {'coordinates': [429.6000061035156,
 

In [60]:
[sent["sent_text"] for sent in sent_dicts_with_coords if sent["tokens_data"][-1]["ref"]["page"][-1]==4]

['Ego uero nec Quercetanum in magna dignitate apud Gallorum regem constitutum, medicumque & philosophum parabolicum quidem, sed minime stultum aut indoctum, id quod in publicum edita scripta, & plurium doctorum testimonia hactenus comprobarunt, meo calculo ausim aut uelim de gradum deiicere, nec eius medicinam & physiologiam, licet non undiquaque ad sensus meos compositam, contumelia aut dedecore afficere, cuius consilii ratio non obscure podaeque & ratio & familia.',
 'Primam appellant sectam dogmaticorum & rationalium, cuius professores ab Hippocrate & Galeno Hippocratici & Galenici nuncupantur, quia potissmum, istorum, instituta sequuntur.',
 'Quidam adiiciunt & Arabum doctrinam, quanquam non omnem, & sic in diuersas factiones etiam alias hoc genus diuiditur, ex quibus quaedam etiam Galeni placitis non omnino acquiescunt.',
 'Sed hoc palam est.',
 'Iuxta praesentem considerationem, Galenici sunt duplices:',
 'Quidam seruiles, qui nihil ne latum quidem unguem a dogmatibus Galeni sibi

In [57]:
sent_dicts_margins_with_coords[:5]

[{'work_id': '100085',
  'sent_id': 0,
  'sent_text': 'In specie tot medicorum hodie sectae, ut uideantur innumerabila, genere ad tres classes reduci possunt.',
  'tokens_data': [{'token_text': 'In',
    'lemma': 'in',
    'pos': 'ADP',
    'ref': {'page': [4],
     'textblock': [0],
     'tags': None,
     'blocktype': 'margin'},
    'char_start': 0,
    'char_end': 2,
    'coordinates': [235.44000244140625,
     684.7200317382812,
     409.4384460449219,
     689.5200805664062]},
   {'token_text': 'specie',
    'lemma': 'species',
    'pos': 'NOUN',
    'ref': {'page': [4],
     'textblock': [0],
     'tags': None,
     'blocktype': 'margin'},
    'char_start': 3,
    'char_end': 9,
    'coordinates': [235.44000244140625,
     684.7200317382812,
     409.4384460449219,
     689.5200805664062]},
   {'token_text': 'tot',
    'lemma': 'tot',
    'pos': 'DET',
    'ref': {'page': [4],
     'textblock': [0],
     'tags': None,
     'blocktype': 'margin'},
    'char_start': 10,
    'char_e

In [52]:
for sent in sent_dicts_margins[:3]:
    print(f"SENT #{sent['sent_id']}: {sent['sent_text']}")
    for tok in sent["tokens_data"]:
        coords = get_token_coordinates(tok, textblocks)
        print("  ", tok["token_text"], coords)

SENT #0: In specie tot medicorum hodie sectae, ut uideantur innumerabila, genere ad tres classes reduci possunt.
   In [235.44000244140625, 684.7200317382812, 409.4384460449219, 689.5200805664062]
   specie [235.44000244140625, 684.7200317382812, 409.4384460449219, 689.5200805664062]
   tot [235.44000244140625, 684.7200317382812, 409.4384460449219, 689.5200805664062]
   medicorum [235.44000244140625, 725.0398559570312, 410.12158203125, 729.8399047851562]
   hodie [235.44000244140625, 765.8640747070312, 409.9117736816406, 770.7840576171875]
   sectae [235.44000244140625, 765.8640747070312, 409.9117736816406, 770.7840576171875]
   , [235.44000244140625, 765.8640747070312, 409.9117736816406, 770.7840576171875]
   ut [240.24000549316406, 806.3999633789062, 399.8396301269531, 811.2000122070312]
   uideantur [240.24000549316406, 806.3999633789062, 399.8396301269531, 811.2000122070312]
   innumerabila [237.83999633789062, 846.2639770507812, 409.64459228515625, 851.1839599609375]
   , [237.839

In [113]:
altblocks = []
for n, sent in enumerate(sent_dicts):
    for token in sent["tokens_data"]:
        if token["ref"]["blocktype"] == "margin":
            altblocks.append([sent_dicts[n -1], sent])
            break

In [114]:
len(altblocks)

62

In [115]:
altblocks

[[{'work_id': '100085',
   'sent_id': 28,
   'sent_text': 'Ego uero nec Quercetanum in magna dignitate apud Gallorum regem constitutum, medicumque & philosophum parabolicum quidem, sed minime stultum aut indoctum, id quod in publicum edita scripta, & plurium doctorum testimonia hactenus comprobarunt, meo calculo ausim aut uelim de gradum deiicere, nec eius medicinam & physiologiam, licet non undiquaque ad sensus meos compositam, contumelia aut dedecore afficere, cuius consilii ratio non obscure',
   'tokens_data': [{'token_text': 'Ego',
     'lemma': 'ego',
     'pos': 'PRON',
     'ref': {'page': [3],
      'textblock': [27],
      'tags': None,
      'blocktype': 'text'},
     'char_start': 0,
     'char_end': 3},
    {'token_text': 'uero',
     'lemma': 'uero',
     'pos': 'CCONJ',
     'ref': {'page': [3],
      'textblock': [27],
      'tags': None,
      'blocktype': 'text'},
     'char_start': 4,
     'char_end': 8},
    {'token_text': 'nec',
     'lemma': 'neque',
     'pos': '

In [107]:
def process_file_to_v2_json(filepath, work_id, outpath, nlp):
    """
    Load OCR textblocks → build Doc → convert to v2 sentence dicts → write JSON.
    """

    # Load textblocks
    with open(filepath, 'r', encoding='utf-8') as f:
        textblocks_pages = json.load(f)

    # Build annotated doc
    doc = process_with_source_tracking(textblocks_pages, nlp)

    # Convert doc to sentence dictionaries
    sent_dicts = doc_to_sent_dicts(doc, work_id)

    # Write JSON
    with open(outpath, "w", encoding="utf-8") as f:
        json.dump(sent_dicts, f, ensure_ascii=False, indent=2)

In [ ]:
source_path = "../data/emlap_annotated_textblocks/"
target_path = "../data/sents_data_jsons_dicts/"
os.makedirs(target_path, exist_ok=True)

for filename in os.listdir(source_path):

    if not filename.endswith(".json"):      # textblocks files
        continue

    work_id = filename[:6]
    outpath = os.path.join(target_path, f"{work_id}.json")

    # Skip if already processed
    if os.path.exists(outpath):
        continue

    filepath = os.path.join(source_path, filename)
    print("Processing", filename)

    try:
        process_file_to_v2_json(
            filepath=filepath,
            work_id=work_id,
            outpath=outpath,
            nlp=tomela.nlp
        )
    except Exception as e:
        print("FAILED:", filename, "→", e)

In [98]:
sent_data_updated[100:120]

[('100085',
  100,
  'Non alium liquorem uesica reddunt.',
  [('Non',
    'non',
    'PART',
    (0, 3),
    {'page': [7], 'texblock': [23], 'tags': None, 'blocktype': 'text'}),
   ('alium',
    'alius',
    'DET',
    (4, 9),
    {'page': [7], 'texblock': [23], 'tags': None, 'blocktype': 'text'}),
   ('liquorem',
    'liquor',
    'NOUN',
    (10, 18),
    {'page': [7], 'texblock': [23], 'tags': None, 'blocktype': 'text'}),
   ('uesica',
    'uesica',
    'NOUN',
    (19, 25),
    {'page': [7], 'texblock': [23, 24], 'tags': None, 'blocktype': 'text'}),
   ('reddunt',
    'reddo',
    'VERB',
    (26, 33),
    {'page': [7], 'texblock': [24], 'tags': None, 'blocktype': 'text'}),
   ('.',
    '.',
    'PUNCT',
    (33, 34),
    {'page': [7], 'texblock': [24], 'tags': None, 'blocktype': 'text'})]),
 ('100085',
  101,
  'Non habent alia membra, quam nos, non sudores alios.',
  [('Non',
    'non',
    'PART',
    (0, 3),
    {'page': [7], 'texblock': [24], 'tags': None, 'blocktype': 'text'}

In [ ]:
    target_path = "/srv/data/tome/tome-corpus/sents_data_id_jsons_v4-0/"
os.makedirs(target_path, exist_ok=True)

In [ ]:
# filename_id_dict.items()

In [ ]:
source_path = "../data/emlap_annotated_textblocks/"
len(os.listdir(source_path)) # 100 for textblocks, 100 for parameters used for their extraction

In [ ]:
os.listdir(source_path)

In [ ]:
for filename in os.listdir(source_path):
    if "_params" not in filename:
            id = filename[:6]
            try:
                if id + ".json" not in os.listdir(target_path):
                    # filename = filename.replace(".pdf", ".json")
                    filepath = os.path.join(source_path, filename)
                    with open(filepath, 'r', encoding='utf-8') as f:
                            textblocks_pages = json.load(f)
                    print("currently processing: ", filename)
                    doc = process_with_source_tracking(textblocks_pages, tomela.nlp)
                    doc_sentdata = [(sent.text, [(t.text, t.lemma_, t.pos_, (t.idx - sent[0].idx, t.idx - sent[0].idx + len(t)),  {"page" : t._.pages, "texblock" : t._.textblocks}) for t in sent]) for sent in doc.sents]
                    sent_data_updated = []
                    for n_sent, sent_data in enumerate(doc_sentdata):
                            sent_data_updated.append((id, n_sent, sent_data[0], sent_data[1]))
                    with open(target_path + str(id) + ".json", "w") as f:
                            json.dump(sent_data_updated, f)
            except:
                print("failed with file: ", id, filename)
                pass

In [ ]:
fns_jsons = os.listdir(target_path)
fns_jsons[:10]

In [ ]:
len(fns_jsons)

In [ ]:
sents_data = json.load(open(target_path + fns_jsons[20], "r"))
sents_data[100:103]

In [ ]:
import os, json, pickle
prev_path   = target_path  # where your old .pickle / .json live
target_path = "../data/sents_data_jsons_dicts/"
os.makedirs(target_path, exist_ok=True)

def token_tuple_to_dict(tok):
    """
    Accepts token tuples of length 4 or 5:
      4: (text, lemma, pos, (start, end))
      5: (text, lemma, pos, (start, end), ref_dict)
    Returns a JSON-serializable dict.
    """
    if len(tok) < 4:
        raise ValueError(f"Unexpected token shape: {tok}")

    token_text, lemma, pos, span = tok[0], tok[1], tok[2], tok[3]
    if not isinstance(span, (list, tuple)) or len(span) != 2:
        raise ValueError(f"Bad span in token: {tok}")

    char_start, char_end = int(span[0]), int(span[1])

    # Optional ref at index 4
    ref = tok[4] if len(tok) >= 5 else None

    # Make sure ref is JSON-friendly
    if isinstance(ref, dict):
        page = ref.get("page")
        # convert sets/tuples to lists to be JSON-serializable
        if isinstance(page, (set, tuple)):
            page = list(page)
        ref = {
            "page": page,
            "textblock": ref.get("textblock") or ref.get("texblock")  # tolerate earlier key name
        }
    elif ref is not None:
        # unexpected type → stringify to avoid JSON errors
        ref = str(ref)

    return {
        "token_text": token_text,
        "lemma": lemma,
        "pos": pos,
        "ref": ref,                  # << stays None if we don’t have it
        "char_start": char_start,
        "char_end": char_end,
    }

def sent_tuple_to_dict(entry):
    """
    entry is typically: (work_id, sent_id, sent_text, tokens_list)
    """
    if len(entry) != 4:
        raise ValueError(f"Unexpected sentence shape: {type(entry)} {entry}")

    work_id, sent_id, sent_text, tokens_list = entry
    tokens_dicts = [token_tuple_to_dict(tok) for tok in tokens_list]
    return {
        "work_id": work_id,
        "sent_id": int(sent_id),
        "sent_text": sent_text,
        "tokens_data": tokens_dicts,
    }

def load_any(path):
    """
    Load .pickle OR .json produced by your previous run.
    Must return a list of sentence entries (tuples/lists).
    """
    if path.endswith(".pickle"):
        with open(path, "rb") as f:
            return pickle.load(f)
    elif path.endswith(".json"):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    else:
        raise ValueError(f"Unsupported file type: {path}")

for fn in os.listdir(prev_path):
    if not (fn.endswith(".pickle") or fn.endswith(".json")):
        continue

    # Keep your 6-char doc id convention if you like
    doc_id = fn[:6]
    out_path = os.path.join(target_path, f"{doc_id}.json")
    if os.path.exists(out_path):
        continue

    try:
        prev_data = load_any(os.path.join(prev_path, fn))
        # If the previous version stored only (sent_text, tokens) per sentence,
        # reconstruct work_id/sent_id here as needed:
        # e.g., prev_data == [(sent_text, tokens), ...]
        if prev_data and len(prev_data[0]) == 2:
            # synthesize (work_id, sent_id, sent_text, tokens)
            prev_data = [(doc_id, i, s[0], s[1]) for i, s in enumerate(prev_data)]

        sents_dicts = [sent_tuple_to_dict(row) for row in prev_data]

        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(sents_dicts, f, ensure_ascii=False, indent=2)

        print("wrote:", out_path)

    except Exception as e:
        print("failed:", fn, "-", e)

In [32]:
doc, doc_margins = process_with_source_tracking(textblocks[:20], tomela.nlp)

In [ ]:
lemmatized_sents_path = "/srv/data/tome/tome-corpus/lemmatized_sents_v4-0/"
try:
    os.mkdir(lemmatized_sents_path)
except:
    pass

In [ ]:
os.listdir(lemmatized_sents_path)

In [ ]:
for fn in fns_jsons:
    lemmatized_sents = []
    sents_data = json.load(open(target_path + fn, "rb"))
    print(fn)
    for (doc_id, sent_id, sent_text, sent_data) in sents_data:
        lemmasent = []
        for wordform, lemma, tag, position, t_ref in sent_data:
            if tag in ["NOUN", "PROPN", "ADJ", "VERB"]:
                lemmasent.append(lemma.lower())
        lemmatized_sents.append(" ".join(lemmasent) + "\n")
    with open(lemmatized_sents_path + fn.replace(".json", ".txt"), "w", encoding="utf-8") as f:
        f.writelines(lemmatized_sents)

In [ ]:
import shutil
shutil.copytree("/srv/data/tome/tome-corpus/lemmatized_sents_v4-0/", "../data/lemmatized_sents", dirs_exist_ok=True)
shutil.copytree("/srv/data/tome/tome-corpus/sents_data_id_jsons_v4-0/", "../data/sents_data", dirs_exist_ok=True)